In [1]:
!pip install -q pypdf
!pip install -q transformers einops accelerate langchain langchain_experimental bitsandbytes
!pip install -q sentence_transformers
!pip install -q llama_index llama-index
!pip install -q llama-index llama-index-readers-file pypdf
!pip install -U langchain langchain-community sentence-transformers
!pip install -Uq llama-index-llms-huggingface
!pip install -U langchain-huggingface sentence-transformers
!pip install bitsandbytes accelerate
!pip install llama-index-embeddings-langchain
!pip install -U unstructured unstructured[pdf] pathlib
!apt-get update --fix-missing
!apt-get install -y poppler-utils
!pip install qdrant-client llama-index-vector-stores-qdrant

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 30.9 MB/s eta 0:00:00
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-1.19.0-py3-none-any.whl (693 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-1.19.0-py3-none-any.whl (693 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.19.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.5 MB/s eta 0:0

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,303 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InReleas

In [2]:
from pathlib import Path

from llama_index.readers.file import UnstructuredReader
from llama_index.core.schema import TextNode
from llama_index.core import VectorStoreIndex, StorageContext, Settings

from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.langchain import LangchainEmbedding

from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

from qdrant_client import QdrantClient

reader = UnstructuredReader()

files_path = Path('/content/data')

documents = []

for file in files_path.rglob("*.pdf"):
  docs = reader.load_data(file=str(file))
  for d in docs:
    d.metadata["file_name"] = file.name
  documents.extend(docs)

print(f"Loaded {len(documents)} raw document parts")


/tmp/ipykernel_19963/3531540625.py:10: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Loaded 5 raw document parts


In [3]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

embed_model = LangchainEmbedding(hf_embeddings)


/tmp/ipykernel_19963/2573348723.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
splitter = SemanticChunker(
    hf_embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85
)

all_chunks = []

for doc in documents:
    chunks = splitter.create_documents([doc.text])

    for c in chunks:
        c.metadata = dict(doc.metadata)

    all_chunks.extend(chunks)

print(f"Total semantic chunks: {len(all_chunks)}")

In [ ]:
for i, c in enumerate(all_chunks[:3]):
    print("\n" + "="*60)
    print(c.metadata)
    print(c.page_content[:300])

In [ ]:
from llama_index.core.schema import TextNode

nodes = [
    TextNode(
        text=chunk.page_content,
        metadata={
            **chunk.metadata,
            "chunk_idx": i
        }
    )
    for i, chunk in enumerate(all_chunks)
]

print(f"Converted {len(nodes)} chunks to TextNodes")




In [ ]:
client = QdrantClient(":memory:")  # change to path="..." or url="..." for production

vector_store = QdrantVectorStore(
    client=client,
    collection_name="pdf_chunks"
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

In [ ]:
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.huggingface import HuggingFaceLLM
# System prompt — instructions given to the model before every question
system_prompt = """
You are a Q&A assistant.

Rules:
- Answer ONLY using the provided context.
- Be concise (max 3-5 sentences).
- DO NOT repeat the answer.
- STOP immediately after giving the final answer.
- Do not add phrases like "The answer is:".
"""

QA_PROMPT = PromptTemplate(
"""You are a precise assistant.

Use ONLY the context below.
Do NOT repeat answers.
Do NOT include "Answer:" labels.
Write only one final response.

Context:
{context_str}

Question:
{query_str}

Final answer:"""
)

llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=150,
    generate_kwargs={
        "temperature": 0.0,
        "do_sample": False,
        "repetition_penalty": 1.2,   # 🔥 IMPORTANT FIX
        "top_p": 1.0
    },
    tokenizer_name="Qwen/Qwen3-0.6B",
    model_name="Qwen/Qwen3-0.6B",
    device_map="auto"
)

print("LLM loaded successfully!")

In [ ]:
Settings.llm = llm
Settings.embed_model = embed_model

In [ ]:
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model
)

print("Index built successfully in Qdrant")


In [ ]:
query_engine = index.as_query_engine(
    text_qa_template=QA_PROMPT,
    similarity_top_k=2,
    response_mode="compact"
)


print("Query engine ready")

In [ ]:
def ask(question):
    print(f"\n❓ {question}")
    response = query_engine.query(question)
    print(f"\n🤖 {response}")
    print("\n📚 Sources:")
    for node in response.source_nodes:
        print(f"  score={node.score:.3f} | {node.text[:150]}...")



In [ ]:
print(ask("What exactly is LOA and what is needed for application?"))

In [ ]:
from llama_index.vector_stores.qdrant import QdrantVectorStore

vector_store = QdrantVectorStore(
    client=client,
    collection_name="pdf_chunks"
)

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex


storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

In [ ]:
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from llama_index.core.prompts import PromptTemplate

# System prompt — instructions given to the model before every question
system_prompt = """
You are a Q&A assistant.

Rules:
- Answer ONLY using the provided context.
- Be concise (max 3-5 sentences).
- DO NOT repeat the answer.
- STOP immediately after giving the final answer.
- Do not add phrases like "The answer is:".
"""

QA_PROMPT = PromptTemplate(
    "<|SYSTEM|>" + system_prompt + "\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer (no repetition, no prefix):"
)

llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=256,   # reduce this too
    generate_kwargs={
    "temperature": 0.0,
    "do_sample": False
},
    tokenizer_name="Qwen/Qwen3-0.6B",
    model_name="Qwen/Qwen3-0.6B",
    device_map="auto"
)

print("LLM loaded successfully!")

In [ ]:
from llama_index.embeddings.langchain import LangchainEmbedding

embed_model = LangchainEmbedding(embeddings)

In [ ]:
from llama_index.core import VectorStoreIndex, Settings

Settings.llm = llm
Settings.embed_model = embed_model

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context
)

index.storage_context.persist(persist_dir="./saved_index")

print("Index built and saved.")

In [ ]:
query_engine = index.as_query_engine(
    text_qa_template=QA_PROMPT,
    similarity_top_k=3,
    response_mode="refined"
)

In [ ]:
def ask(question):
    print(f"\n❓ {question}")
    response = query_engine.query(question)
    print(f"\n🤖 {response}")
    print("\n📚 Sources:")
    for node in response.source_nodes:
        print(f"  score={node.score:.3f} | {node.text[:150]}...")



In [ ]:
ask("What exactly is Etisalat BQS?")